# ĐỒ ÁN TỐT NGHIỆP: YoFaSoShop_VTON (Ứng dụng TMĐT Thời trang tích hợp Thử đồ ảo)
## NOTEBOOK THỬ NGHIỆM TUẦN 4 - VER2 OOTDIFFUSION-ONLY (21/09 - 27/09/2026)
**Sinh viên thực hiện:** Nguyễn Vũ Khanh - MSSV: 22115053122118 - Lớp: 22T1  
**Giảng viên hướng dẫn:** ThS. Phạm Thị Trà My - Khoa Công nghệ số, ĐH Sư phạm Kỹ thuật - ĐH Đà Nẵng  

---
### Mục tiêu công việc Tuần 4:
1. Chạy thử nghiệm bóc tách phông nền sản phẩm shop bằng **DeepFashion2 (YOLOv8-seg)**, lưu ảnh tách nền vào `results/segmented/` và trích xuất bảng màu chủ đạo (Color Palette).
2. Chạy chuỗi thử đồ ảo **OOTDiffusion VITON-HD** cho `upper_body` và **OOTDiffusion DressCode** cho `lower_body`/`dress` qua GPU đám mây.
3. Xuất và lưu trữ **2 định dạng kết quả** vào `results/tryon/`:
   - **Ảnh thử đồ đơn (Single Try-on):** Chỉ chứa người mẫu sau khi đã ướm trang phục mới (`tryon_result.png`).
   - **Bảng đối sánh 3 cột:** `[Người mẫu gốc | Sản phẩm bóc nền | Kết quả thử đồ AI]` (`week4_report_comparison.jpg`).
4. Tự động đóng gói và tải kết quả về máy tính cục bộ khi chạy trên Google Colab.


### Bước 1: Kiểm tra GPU Google Colab

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### Bước 2: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q ultralytics diffusers transformers accelerate onnxruntime opencv-python scikit-learn pillow matplotlib huggingface_hub gradio_client


### Bước 3: Tải trọng số DeepFashion2 YOLOv8-seg và khởi tạo thư mục dữ liệu


In [ ]:
import os, sys
from pathlib import Path

# 1. Tự động định vị Root directory chuẩn của Repo (Hỗ trợ cả Google Colab và Local VS Code)
curr_dir = os.getcwd()
if curr_dir.endswith(os.path.join("ml-training", "notebooks")) or curr_dir.endswith("notebooks"):
    # Đang mở từ thư mục notebooks cục bộ trên Windows/Linux -> Chuyển về gốc Repo
    project_root = Path(curr_dir).resolve().parents[1]
    os.chdir(project_root)
    print(f"[*] Đã chuyển thư mục làm việc về gốc Repo: {os.getcwd()}")
else:
    print(f"[*] Thư mục làm việc hiện tại: {os.getcwd()}")

# 2. Khởi tạo cấu trúc cây thư mục chuẩn cho cả Colab và Local
dirs_to_create = [
    "data/test_samples/garments/tops",
    "data/test_samples/garments/bottoms",
    "data/test_samples/garments/dresses",
    "data/test_samples/persons",
    "data/test_samples/results/segmented",
    "data/test_samples/results/tryon",
    "weights"
]
for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)
print("✅ Các thư mục lưu trữ dữ liệu và kết quả đã sẵn sàng!")

# 3. Tải trọng số YOLOv8s-seg DeepFashion2 nếu chưa có
weight_path = "weights/deepfashion2_yolov8s-seg.pt"
if not os.path.exists(weight_path) or os.path.getsize(weight_path) < 100000:
    print("Đang tải trọng số YOLOv8s-seg DeepFashion2 (~24MB)...")
    !wget -q -O weights/deepfashion2_yolov8s-seg.pt https://huggingface.co/Bingsu/adetailer/resolve/main/deepfashion2_yolov8s-seg.pt
    print("Tải hoàn tất trọng số!")
else:
    print("Trọng số YOLOv8s-seg đã sẵn sàng!")


### Bước 4: Upload và kiểm tra ảnh trang phục/người mẫu (chấp nhận tên file tùy ý)


In [ ]:
import io
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

# Bước 4 tự tạo thư mục để vẫn chạy được nếu runtime Colab vừa khởi động lại.
garment_dir = Path("data/test_samples/garments/uploaded")
person_dir = Path("data/test_samples/persons/uploaded")
garment_dir.mkdir(parents=True, exist_ok=True)
person_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("Cell upload này cần chạy trên Google Colab.") from exc

print("[*] Đang nạp mô hình kiểm tra ảnh...")
fashion_detector = YOLO("weights/deepfashion2_yolov8s-seg.pt")
person_detector = YOLO("yolov8n-pose.pt")  # Dùng keypoint để tránh nhầm quần thành người.


def read_uploaded_image(filename, file_bytes):
    """Đọc và chuẩn hóa mọi định dạng ảnh Pillow hỗ trợ sang RGB."""
    try:
        with Image.open(io.BytesIO(file_bytes)) as probe:
            probe.verify()
        image = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        return None, f"'{filename}' không phải file ảnh hợp lệ ({exc})"

    if image.width < 128 or image.height < 128:
        return None, f"'{filename}' quá nhỏ ({image.width}x{image.height}); cần ít nhất 128x128 px"
    return image, None


POSE_KEYPOINT_THRESHOLD = 0.35


def classify_person_frame(keypoint_scores):
    """Phân loại khung người từ 17 keypoint COCO của YOLO Pose."""
    if keypoint_scores is None or len(keypoint_scores) < 17:
        return "invalid", {"shoulders": 0, "hips": 0, "knees": 0, "ankles": 0}

    visible = keypoint_scores >= POSE_KEYPOINT_THRESHOLD
    counts = {
        "shoulders": int(visible[[5, 6]].sum()),
        "hips": int(visible[[11, 12]].sum()),
        "knees": int(visible[[13, 14]].sum()),
        "ankles": int(visible[[15, 16]].sum()),
    }

    upper_core_visible = counts["shoulders"] >= 1 and counts["hips"] >= 1
    knees_visible = counts["knees"] >= 1
    ankles_visible = counts["ankles"] >= 1
    if upper_core_visible and knees_visible and ankles_visible:
        return "full_body", counts
    # Khung 3/4 thấy hông và ít nhất một đầu gối nhưng thiếu mắt cá chân.
    if upper_core_visible and knees_visible:
        return "three_quarter", counts
    # Half-body vẫn hợp lệ cho upper-body; dress có thể chạy ở chế độ giới hạn/cảnh báo.
    if upper_core_visible:
        return "half_body", counts
    return "invalid", counts


def best_person_detection(image):
    result = person_detector.predict(
        source=np.asarray(image), classes=[0], conf=0.25, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0 or result.keypoints is None:
        return 0.0, 0.0, 0, "invalid", {}

    confidences = result.boxes.conf.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()
    image_area = image.width * image.height
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    best_index = int(np.argmax(confidences))
    keypoint_confidences = result.keypoints.conf
    visible_keypoints = 0
    person_frame = "invalid"
    frame_keypoints = {}
    if keypoint_confidences is not None:
        keypoint_scores = keypoint_confidences[best_index].cpu().numpy()
        visible_keypoints = int((keypoint_scores >= POSE_KEYPOINT_THRESHOLD).sum())
        person_frame, frame_keypoints = classify_person_frame(keypoint_scores)
    return (
        float(confidences[best_index]),
        float(areas[best_index] / image_area),
        visible_keypoints,
        person_frame,
        frame_keypoints,
    )


def best_garment_detection(image):
    result = fashion_detector.predict(
        source=np.asarray(image), conf=0.25, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0

    confidences = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)
    best_index = int(np.argmax(confidences))
    class_id = int(classes[best_index])
    label = result.names[class_id] if isinstance(result.names, dict) else result.names[class_id]
    return str(label), float(confidences[best_index])


def validate_image_role(image, expected_role):
    if expected_role == "person":
        (
            person_confidence,
            person_area_ratio,
            visible_keypoints,
            person_frame,
            frame_keypoints,
        ) = best_person_detection(image)
        frame_summary = ", ".join(
            f"{name}={count}/2" for name, count in frame_keypoints.items()
        ) or "không có keypoint"
        if (
            person_confidence < 0.35
            or person_area_ratio < 0.08
            or visible_keypoints < 5
            or person_frame == "invalid"
        ):
            return False, (
                "không nhận diện được người đủ rõ "
                f"(confidence={person_confidence:.2f}, vùng người={person_area_ratio:.0%}, "
                f"keypoints={visible_keypoints}/17, {frame_summary}). "
                "Hãy dùng ảnh chính diện và thấy rõ ít nhất từ vai đến hông. "
                "Mức phù hợp với từng loại garment sẽ được kiểm tra trước khi try-on"
            ), None
        return True, (
            f"đã nhận diện person (confidence={person_confidence:.2f}, "
            f"keypoints={visible_keypoints}/17, frame={person_frame}, {frame_summary})"
        ), person_frame

    garment_label, garment_confidence = best_garment_detection(image)
    if garment_label is None or garment_confidence < 0.25:
        return False, "không nhận diện được quần áo bằng DeepFashion2", None
    return True, (
        f"đã nhận diện garment '{garment_label}' "
        f"(confidence={garment_confidence:.2f}); Bước 5 sẽ tách riêng trang phục bằng mask"
    ), garment_label


def safe_stem(filename):
    stem = re.sub(r"[^0-9A-Za-z_-]+", "_", Path(filename).stem).strip("_")
    return stem[:80] or "uploaded_image"


def upload_validated_image(expected_role, destination_dir):
    role_name = "TRANG PHỤC" if expected_role == "garment" else "NGƯỜI MẪU"
    print(f"\n👉 Chọn ảnh {role_name}; tên file có thể tùy ý (PNG/JPG/JPEG/WEBP...):")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError(f"Chưa có file nào được chọn cho {role_name}.")

    rejected = []
    for filename, file_bytes in uploaded.items():
        image, image_error = read_uploaded_image(filename, file_bytes)
        if image_error:
            print(f"❌ {image_error}")
            rejected.append(filename)
            continue

        accepted, message, detected_label = validate_image_role(image, expected_role)
        if not accepted:
            print(f"❌ Từ chối '{filename}': {message}.")
            rejected.append(filename)
            continue

        destination = destination_dir / f"{expected_role}_{safe_stem(filename)}.png"
        image.save(destination, format="PNG")
        print(f"✅ Chấp nhận '{filename}': {message}.")
        print(f"   Đã chuẩn hóa và lưu tại: {destination}")
        return str(destination), detected_label

    raise ValueError(
        f"Không có ảnh {role_name} hợp lệ. Hãy chọn đúng loại ảnh rồi chạy lại Bước 4. "
        f"Các file đã từ chối: {', '.join(rejected)}"
    )


# Mỗi lần chạy cell, người dùng có thể chọn ảnh mới với bất kỳ tên file nào.
garment_path, garment_class_name = upload_validated_image("garment", garment_dir)
person_path, person_frame = upload_validated_image("person", person_dir)
garment_description = (garment_class_name or "clothing").replace("_", " ")

UPPER_BODY_LABELS = {
    "short_sleeved_shirt",
    "long_sleeved_shirt",
    "short_sleeved_outwear",
    "long_sleeved_outwear",
    "vest",
    "sling",
}
LOWER_BODY_LABELS = {"shorts", "trousers", "skirt"}
DRESS_LABELS = {
    "short_sleeved_dress",
    "long_sleeved_dress",
    "vest_dress",
    "sling_dress",
}
SUPPORTED_GARMENT_LABELS = UPPER_BODY_LABELS | LOWER_BODY_LABELS | DRESS_LABELS


def garment_category_from_label(label):
    normalized = (label or "").strip().lower()
    if normalized in UPPER_BODY_LABELS:
        return "upper_body"
    if normalized in LOWER_BODY_LABELS:
        return "lower_body"
    if normalized in DRESS_LABELS:
        return "dress"
    return None

garment_category = garment_category_from_label(garment_class_name)
if garment_category is None:
    raise ValueError(
        f"Nhãn garment '{garment_class_name}' không thuộc 13 nhãn DeepFashion2 được hỗ trợ. "
        "Notebook đã dừng trước khi gọi GPU."
    )

print("\n--- THÔNG TIN DỮ LIỆU ĐÃ KIỂM TRA THÀNH CÔNG ---")
print(f"[+] Trang phục: {garment_path} ({os.path.getsize(garment_path)/1024:.1f} KB)")
print(f"[+] Loại đồ   : {garment_description}")
print(f"[+] Vùng try-on: {garment_category}")
print(f"[+] Người mẫu : {person_path} ({os.path.getsize(person_path)/1024:.1f} KB)")
print(f"[+] Khung người: {person_frame}")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(Image.open(garment_path))
ax[0].set_title(f"Trang phục: {Path(garment_path).name}", fontsize=12)
ax[0].axis("off")
ax[1].imshow(Image.open(person_path))
ax[1].set_title(f"Người mẫu: {Path(person_path).name}\nKhung: {person_frame}", fontsize=12)
ax[1].axis("off")
plt.tight_layout()
plt.show()


### Bước 5: Chạy thử nghiệm bóc tách nền sản phẩm (DeepFashion2 YOLOv8-seg & Color Palette)


In [ ]:
from ultralytics import YOLO
import cv2, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

model = YOLO("weights/deepfashion2_yolov8s-seg.pt")
print("Nạp YOLOv8s-seg thành công!")
print(f"Đang phân đoạn và tách nền: {garment_path}")

# Đảm bảo thư mục lưu trữ tồn tại
out_segmented_dir = "data/test_samples/results/segmented"
os.makedirs(out_segmented_dir, exist_ok=True)
out_nobg_path = os.path.join(out_segmented_dir, "garment_nobg.png")
out_mask_path = os.path.join(out_segmented_dir, "garment_mask.png")
out_isolated_path = os.path.join(out_segmented_dir, "garment_isolated_white.png")

# Dự đoán bằng YOLOv8-seg DeepFashion2
results = model.predict(source=garment_path, conf=0.25, verbose=False)
res = results[0]

orig_img = Image.open(garment_path).convert("RGB")
orig_np = np.array(orig_img)
h, w = orig_np.shape[:2]

if res.boxes is None or len(res.boxes) == 0:
    raise ValueError("DeepFashion2 không còn nhận diện được garment ở Bước 5. Hãy chạy lại Bước 4.")

box_confidences = res.boxes.conf.cpu().numpy()
box_classes = res.boxes.cls.cpu().numpy().astype(int)
detected_labels = [str(res.names[int(class_id)]) for class_id in box_classes]
matching_indices = [
    index for index, label in enumerate(detected_labels)
    if label == garment_class_name
]
candidate_indices = matching_indices or list(range(len(box_confidences)))
selected_index = max(candidate_indices, key=lambda index: float(box_confidences[index]))
selected_label = detected_labels[selected_index]
selected_confidence = float(box_confidences[selected_index])
garment_class_name = selected_label
garment_description = selected_label.replace("_", " ")
garment_category = garment_category_from_label(selected_label)
if garment_category is None:
    raise ValueError(
        f"Mask được chọn có nhãn không hỗ trợ '{selected_label}'. "
        "Notebook đã dừng trước khi gọi GPU."
    )
print(f"[*] Chọn mask '{selected_label}' có confidence={selected_confidence:.2f}")
print(f"[*] Định tuyến try-on theo vùng: {garment_category}")

if res.masks is not None and len(res.masks) > selected_index:
    mask_tensor = res.masks.data[selected_index].cpu().numpy()
    mask_resized = cv2.resize((mask_tensor * 255).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    binary_mask = (mask_resized > 127).astype(np.uint8)
    mask_y, mask_x = np.where(binary_mask > 0)
    if len(mask_x) == 0 or len(mask_y) == 0:
        raise ValueError("Mask DeepFashion2 rỗng; không thể tách trang phục.")

    # Crop sát garment để loại người/nền khỏi đầu vào OOTDiffusion.
    padding = max(8, int(0.03 * max(w, h)))
    x1 = max(0, int(mask_x.min()) - padding)
    x2 = min(w, int(mask_x.max()) + padding + 1)
    y1 = max(0, int(mask_y.min()) - padding)
    y2 = min(h, int(mask_y.max()) + padding + 1)
    
    # Tạo ảnh RGBA trong suốt, làm mềm biên bằng Gaussian Blur
    rgba_full = np.zeros((h, w, 4), dtype=np.uint8)
    rgba_full[:, :, :3] = orig_np
    rgba_full[:, :, 3] = cv2.GaussianBlur(binary_mask * 255, (5, 5), 0)
    rgba = rgba_full[y1:y2, x1:x2]
    Image.fromarray(rgba).save(out_nobg_path)
    cv2.imwrite(out_mask_path, (binary_mask * 255).astype(np.uint8))

    alpha = rgba[:, :, 3:4].astype(np.float32) / 255.0
    isolated_tight = Image.fromarray(
        (rgba[:, :, :3] * alpha + 255 * (1 - alpha)).astype(np.uint8)
    )

    # OOTDiffusion luôn resize garment về 768x1024. Đặt garment lên canvas 3:4 trước
    # để giữ nguyên tỷ lệ thật, tránh áo/váy bị kéo méo khi gửi tới Space.
    canvas_width, canvas_height = 768, 1024
    margin_ratio = 0.06
    usable_width = int(canvas_width * (1 - 2 * margin_ratio))
    usable_height = int(canvas_height * (1 - 2 * margin_ratio))
    fit_scale = min(
        usable_width / isolated_tight.width,
        usable_height / isolated_tight.height,
    )
    fitted_size = (
        max(1, int(round(isolated_tight.width * fit_scale))),
        max(1, int(round(isolated_tight.height * fit_scale))),
    )
    resampling = getattr(Image, "Resampling", Image).LANCZOS
    isolated_fitted = isolated_tight.resize(fitted_size, resampling)
    isolated_canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
    paste_xy = (
        (canvas_width - fitted_size[0]) // 2,
        (canvas_height - fitted_size[1]) // 2,
    )
    isolated_canvas.paste(isolated_fitted, paste_xy)
    isolated_canvas.save(out_isolated_path)
    isolated_rgb = np.asarray(isolated_canvas)
    garment_for_tryon_path = out_isolated_path
    print(f"✅ Đã tách riêng garment khỏi người/nền: {os.path.abspath(out_nobg_path)} ({os.path.getsize(out_nobg_path)/1024:.1f} KB)")
    print(f"✅ Đã tạo garment 768x1024, giữ đúng tỷ lệ trên nền trắng: {os.path.abspath(out_isolated_path)}")
    print(f"✅ Đã lưu mask phân đoạn nhị phân: {os.path.abspath(out_mask_path)} ({os.path.getsize(out_mask_path)/1024:.1f} KB)")
    
    # Trích xuất Color Palette (K-Means Clustering 3 màu)
    fg_pixels = orig_np[binary_mask > 0]
    if len(fg_pixels) > 50:
        sample_pixels = fg_pixels[::max(1, len(fg_pixels) // 1000)]
        kmeans = KMeans(n_clusters=min(3, len(sample_pixels)), random_state=42, n_init=5).fit(sample_pixels)
        colors = kmeans.cluster_centers_.astype(int)
        hex_codes = [f"#{c[0]:02X}{c[1]:02X}{c[2]:02X}" for c in colors]
        print(f"🎨 Bảng màu chủ đạo trích xuất: {hex_codes}")
    
    # Hiển thị trực quan 3 khung hình
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(orig_img); ax[0].set_title("1. Ảnh sản phẩm gốc")
    ax[1].imshow(binary_mask, cmap="gray"); ax[1].set_title("2. Mask phân đoạn DeepFashion2")
    ax[2].imshow(isolated_rgb); ax[2].set_title("3. Garment đã tách riêng")
    for a in ax: a.axis("off")
    plt.tight_layout()
    plt.show()
else:
    raise ValueError(
        f"DeepFashion2 nhận diện '{selected_label}' nhưng không tạo được segmentation mask. "
        "Không dùng threshold dự phòng vì có thể giữ lại người trong ảnh."
    )


### Bước 6: Thử đồ bằng OOTDiffusion VITON-HD/DressCode theo category

> Routing thử nghiệm: `upper_body → VITON-HD /process_hd`; `lower_body`, `dress → DressCode /process_dc`. Khung người được chia thành `full_body`, `three_quarter`, `half_body`; dress cho phép cả ba mức nhưng ảnh thiếu chân chạy ở chế độ giới hạn và có cảnh báo. Quần dài vẫn cần full-body; skirt/shorts chấp nhận ảnh 3/4 hoặc half-body ở chế độ giới hạn và có cảnh báo.

> Với `lower_body`, nhãn `skirt/shorts/trousers` do DeepFashion2 đọc từ ảnh output chỉ dùng để ưu tiên và cảnh báo, không chặn việc đổi subtype. Kết quả được ghép bằng mask của đồ cũ và đồ mới để giữ nguyên da chân, mặt, tóc, giày và nền ngoài vùng garment.

> Bộ chọn dùng hai tầng: trước hết giữ nhóm có mức khớp subtype tốt nhất; sau đó chấm điểm tổng hợp gồm màu garment, bảo toàn vùng ngoài garment, bảo toàn vùng cơ thể không được che, độ fitting hình học và confidence detector. Không metric đơn lẻ nào được tự quyết định kết quả.

> **Xác thực Hugging Face:** mở biểu tượng chìa khóa **Secrets** ở thanh trái Colab, tạo secret tên `HF_TOKEN`, dán token quyền `read`, rồi bật **Notebook access**. Không ghi token trực tiếp vào code.


In [ ]:
import time, os
from PIL import Image
import numpy as np
import cv2

from pathlib import Path


def prepare_oot_person(
    source_path,
    output_path,
    target_size=(768, 1024),
):
    image = Image.open(source_path).convert("RGB")

    original_width, original_height = image.size
    target_width, target_height = target_size

    # Scale theo cùng một tỷ lệ, không kéo méo ảnh.
    scale = min(
        target_width / original_width,
        target_height / original_height,
    )

    resized_width = round(original_width * scale)
    resized_height = round(original_height * scale)

    resized = image.resize(
        (resized_width, resized_height),
        Image.Resampling.LANCZOS,
    )

    # Lấy màu trung bình ở bốn góc làm màu padding.
    corners = [
        image.getpixel((0, 0)),
        image.getpixel((original_width - 1, 0)),
        image.getpixel((0, original_height - 1)),
        image.getpixel((original_width - 1, original_height - 1)),
    ]

    background_color = tuple(
        sum(pixel[channel] for pixel in corners) // len(corners)
        for channel in range(3)
    )

    canvas = Image.new(
        "RGB",
        target_size,
        background_color,
    )

    offset_x = (target_width - resized_width) // 2
    offset_y = (target_height - resized_height) // 2

    canvas.paste(resized, (offset_x, offset_y))
    canvas.save(output_path, "PNG")

    metadata = {
        "original_size": (original_width, original_height),
        "content_box": (
            offset_x,
            offset_y,
            offset_x + resized_width,
            offset_y + resized_height,
        ),
        "target_size": target_size,
    }

    return output_path, metadata


def restore_oot_result(output_image, metadata):
    output_image = output_image.convert("RGB")

    # Phòng trường hợp backend trả về kích thước khác.
    if output_image.size != metadata["target_size"]:
        output_image = output_image.resize(
            metadata["target_size"],
            Image.Resampling.LANCZOS,
        )

    # Bỏ vùng padding đã thêm trước inference.
    output_image = output_image.crop(metadata["content_box"])

    # Đưa kết quả trở lại đúng kích thước ảnh người ban đầu.
    return output_image.resize(
        metadata["original_size"],
        Image.Resampling.LANCZOS,
    )

print("=" * 70)
print("KHỞI CHẠY OOTDIFFUSION CHO MỌI CATEGORY")
print(f"[*] Ảnh người mẫu      : {person_path}")
print(f"[*] Ảnh garment đã tách : {garment_for_tryon_path}")
print("=" * 70)

start_time = time.time()
tryon_dir = "data/test_samples/results/tryon"
os.makedirs(tryon_dir, exist_ok=True)
tryon_out_path = os.path.join(tryon_dir, "tryon_result.png")

output = None
tryon_backend = None
remote_error = None

def load_hf_token():
    token = os.environ.get("HF_TOKEN")
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or token
    except Exception:
        # Secret chưa tồn tại/chưa cấp quyền hoặc đang chạy ngoài Colab.
        pass
    return token


hf_token = load_hf_token()
if hf_token:
    print("[*] Đã tìm thấy HF_TOKEN trong Colab Secrets; request sẽ được xác thực.")
else:
    print("[!] Chưa có HF_TOKEN; request ZeroGPU sẽ dùng quota ẩn danh rất thấp.")

def extract_gradio_image_paths(value):
    paths = []
    if isinstance(value, str):
        return [value]
    if isinstance(value, dict):
        for key in ("path", "image"):
            if key in value:
                paths.extend(extract_gradio_image_paths(value[key]))
    if isinstance(value, (tuple, list)):
        for item in value:
            paths.extend(extract_gradio_image_paths(item))
    return list(dict.fromkeys(paths))


def detect_output_garment(image, expected_category):
    """Nhận diện garment và trả cả mask vùng áo/quần; subtype chỉ là tín hiệu heuristic."""
    image = image.convert("RGB")
    image_np = np.asarray(image)
    result = fashion_detector.predict(
        source=image_np, conf=0.20, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0, None

    labels = [str(result.names[int(class_id)]) for class_id in result.boxes.cls.cpu().numpy()]
    confidences = result.boxes.conf.cpu().numpy()
    # matching_indices = [
    #     index for index, label in enumerate(labels)
    #     if garment_category_from_label(label) == expected_category
    # ]
    # if not matching_indices:
    #     return None, 0.0, None
    matching_indices = [
        index for index, label in enumerate(labels)
        if garment_category_from_label(label) == expected_category
    ]

    # Fallback cho Dress: Nếu là đầm màu sáng/kem tiệp màu nền, DeepFashion2 đôi khi
    # gán nhãn 'short_sleeved_shirt' hoặc 'upper_body' nhưng mask trải dài xuống đùi.
    if not matching_indices and expected_category == "dress" and result.masks is not None:
        for idx, (cls_id, conf) in enumerate(zip(result.boxes.cls.cpu().numpy(), confidences)):
            lbl = str(result.names[int(cls_id)])
            # Nếu nhận diện thành áo hoặc chân váy nhưng che phủ diện tích lớn ở thân
            if any(k in lbl for k in ["shirt", "top", "dress", "skirt"]) and conf >= 0.15:
                matching_indices.append(idx)

    if not matching_indices:
        return None, 0.0, None


    best_index = max(matching_indices, key=lambda index: float(confidences[index]))
    # Chỉ lấy mask của detection tốt nhất. Union mọi detection cùng category có thể
    # trộn mask 'skirt' với detection phụ 'trousers' và vô tình cho phép AI sửa cả chân.
    region_mask = np.zeros(image_np.shape[:2], dtype=np.uint8)
    if result.masks is not None and len(result.masks.data) > best_index:
        raw_mask = result.masks.data[best_index].cpu().numpy()
        resized_mask = cv2.resize(
            raw_mask, (image.width, image.height), interpolation=cv2.INTER_NEAREST
        )
        region_mask = (resized_mask > 0.5).astype(np.uint8) * 255
    else:
        x1, y1, x2, y2 = result.boxes.xyxy[best_index].cpu().numpy().astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(image.width, x2), min(image.height, y2)
        region_mask[y1:y2, x1:x2] = 255

    return labels[best_index], float(confidences[best_index]), region_mask


def garment_edit_mask(source_mask, candidate_mask):
    """Hợp mask đồ cũ/mới và làm mềm biên để giới hạn vùng được phép thay đổi."""
    available_masks = [mask for mask in (source_mask, candidate_mask) if mask is not None]
    if not available_masks:
        return None

    edit_mask = np.maximum.reduce(available_masks).astype(np.uint8)
    edit_mask = cv2.dilate(edit_mask, np.ones((25, 25), np.uint8), iterations=1)
    return cv2.GaussianBlur(edit_mask, (0, 0), sigmaX=6, sigmaY=6)


def outside_garment_change_score(candidate_image, source_image, edit_mask):
    """Điểm càng thấp thì mặt, da, chân và nền ngoài vùng thử đồ càng được giữ nguyên."""
    if edit_mask is None:
        return float("inf")
    candidate_np = np.asarray(candidate_image.convert("RGB"), dtype=np.float32)
    source_np = np.asarray(source_image.convert("RGB"), dtype=np.float32)
    outside = edit_mask < 8
    if not np.any(outside):
        return float("inf")
    pixel_difference = np.mean(np.abs(candidate_np - source_np), axis=2)
    return float(np.mean(pixel_difference[outside]))


def change_to_preservation_score(change_score, scale=25.0):
    if not np.isfinite(change_score):
        return 0.0
    return float(np.exp(-max(0.0, change_score) / scale))


def composite_preserving_source(
    candidate_image, source_image, edit_mask, protected_mask=None
):
    candidate_np = np.asarray(candidate_image.convert("RGB"), dtype=np.float32)
    source_np = np.asarray(source_image.convert("RGB"), dtype=np.float32)
    if edit_mask is None:
        alpha = np.ones((*source_np.shape[:2], 1), dtype=np.float32)
    else:
        alpha = edit_mask.astype(np.float32)[:, :, None] / 255.0
    if protected_mask is not None:
        protection = protected_mask.astype(np.float32)[:, :, None] / 255.0
        alpha *= 1.0 - protection
    protected = candidate_np * alpha + source_np * (1.0 - alpha)
    return Image.fromarray(np.clip(protected, 0, 255).astype(np.uint8))


def detect_person_pose_context(image):
    result = person_detector.predict(
        source=np.asarray(image.convert("RGB")), conf=0.20, verbose=False
    )[0]
    if result.boxes is None or len(result.boxes) == 0 or result.keypoints is None:
        return None
    boxes = result.boxes.xyxy.cpu().numpy()
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    best_index = int(np.argmax(areas))
    keypoints = result.keypoints.xy[best_index].cpu().numpy()
    if result.keypoints.conf is None:
        confidences = np.where(np.any(keypoints > 0, axis=1), 1.0, 0.0)
    else:
        confidences = result.keypoints.conf[best_index].cpu().numpy()
    return {
        "box": boxes[best_index].astype(np.float32),
        "keypoints": keypoints.astype(np.float32),
        "confidences": confidences.astype(np.float32),
    }


def upper_body_protected_mask(image_size, pose_context, target_label):
    """Bảo vệ phần cánh tay mà vest/áo ngắn tay không được phép tự vẽ thêm vải."""
    if pose_context is None:
        return None
    label = (target_label or "").lower()
    if "long_sleeved" in label:
        return None

    width, height = image_size
    keypoints = pose_context["keypoints"]
    confidences = pose_context["confidences"]
    if len(keypoints) < 11:
        return None

    shoulder_span = abs(float(keypoints[6][0] - keypoints[5][0]))
    thickness = max(12, min(42, int(round(max(60.0, shoulder_span) * 0.18))))
    start_fraction = 0.15 if any(name in label for name in ("vest", "sling")) else 0.45
    protected = np.zeros((height, width), dtype=np.uint8)

    for shoulder_index, elbow_index, wrist_index in ((5, 7, 9), (6, 8, 10)):
        if min(confidences[[shoulder_index, elbow_index, wrist_index]]) < 0.25:
            continue
        shoulder = keypoints[shoulder_index]
        elbow = keypoints[elbow_index]
        wrist = keypoints[wrist_index]
        start = shoulder + start_fraction * (elbow - shoulder)
        start_xy = tuple(np.rint(start).astype(int))
        elbow_xy = tuple(np.rint(elbow).astype(int))
        wrist_xy = tuple(np.rint(wrist).astype(int))
        cv2.line(protected, start_xy, elbow_xy, 255, thickness, cv2.LINE_AA)
        cv2.line(protected, elbow_xy, wrist_xy, 255, thickness, cv2.LINE_AA)
        cv2.circle(protected, elbow_xy, thickness // 2, 255, -1, cv2.LINE_AA)
        cv2.circle(protected, wrist_xy, thickness // 2, 255, -1, cv2.LINE_AA)

    if not np.any(protected):
        return None
    return cv2.GaussianBlur(protected, (0, 0), sigmaX=4, sigmaY=4)


def protected_region_artifact_score(candidate_image, source_image, protected_mask):
    """Trả (tỷ lệ artefact nặng, điểm bảo toàn) trên vùng da tay cần giữ nguyên."""
    if protected_mask is None:
        return 0.0, 1.0
    candidate_np = np.asarray(candidate_image.convert("RGB"), dtype=np.float32)
    source_np = np.asarray(source_image.convert("RGB"), dtype=np.float32)
    weights = protected_mask.astype(np.float32) / 255.0
    if float(np.sum(weights)) < 1.0:
        return 0.0, 1.0

    channel_difference = np.mean(np.abs(candidate_np - source_np), axis=2)
    source_luma = 0.299 * source_np[:, :, 0] + 0.587 * source_np[:, :, 1] + 0.114 * source_np[:, :, 2]
    candidate_luma = 0.299 * candidate_np[:, :, 0] + 0.587 * candidate_np[:, :, 1] + 0.114 * candidate_np[:, :, 2]

    # Chỉ coi là artefact khi pixel tối sụp mạnh hoặc đổi màu cực lớn.
    # Sai khác ánh sáng/texture nhỏ do diffusion không được phép loại mọi candidate.
    severe_artifact = ((source_luma - candidate_luma) > 50.0) | (channel_difference > 70.0)
    artifact_ratio = float(np.sum(weights * severe_artifact) / np.sum(weights))
    preservation = float(np.exp(-artifact_ratio / 0.08))
    return artifact_ratio, preservation


def garment_fit_score(candidate_mask, pose_context, expected_category, target_label):
    """Heuristic 0..1 phát hiện garment lệch tâm, tràn thân hoặc phình bất thường."""
    if candidate_mask is None or pose_context is None:
        return 0.5
    binary = candidate_mask > 127
    ys, xs = np.where(binary)
    if len(xs) < 50:
        return 0.0

    x1, x2 = float(xs.min()), float(xs.max() + 1)
    y1, y2 = float(ys.min()), float(ys.max() + 1)
    person_x1, person_y1, person_x2, person_y2 = pose_context["box"]
    person_width = max(1.0, float(person_x2 - person_x1))
    center_offset = abs(((x1 + x2) / 2.0) - ((person_x1 + person_x2) / 2.0)) / person_width
    outside_ratio = float(
        np.mean((xs < person_x1 - 5) | (xs > person_x2 + 5) | (ys < person_y1 - 5) | (ys > person_y2 + 5))
    )
    penalty = 0.35 * min(1.0, center_offset / 0.22) + 0.25 * min(1.0, outside_ratio / 0.05)

    if expected_category == "upper_body":
        keypoints = pose_context["keypoints"]
        confidences = pose_context["confidences"]
        if min(confidences[[5, 6, 11, 12]]) >= 0.25:
            shoulder_span = max(1.0, abs(float(keypoints[6][0] - keypoints[5][0])))
            shoulder_y = float(np.mean(keypoints[[5, 6], 1]))
            hip_y = float(np.mean(keypoints[[11, 12], 1]))
            torso_height = max(1.0, hip_y - shoulder_y)
            width_ratio = (x2 - x1) / shoulder_span
            height_ratio = (y2 - y1) / torso_height
            long_sleeved = "long_sleeved" in (target_label or "").lower()
            max_width_ratio = 3.0 if long_sleeved else 1.9
            max_height_ratio = 2.2 if long_sleeved else 1.45
            penalty += 0.20 * min(1.0, max(0.0, width_ratio - max_width_ratio) / 0.8)
            penalty += 0.20 * min(1.0, max(0.0, height_ratio - max_height_ratio) / 0.8)

    return float(np.clip(1.0 - penalty, 0.0, 1.0))

def calculate_limb_integrity_score(candidate_mask, pose_context, garment_label):
    """
    Chấm điểm độ toàn vẹn chi thể (0.0 - 1.0):
    Phát hiện và phạt nặng candidate nếu áo try-on phình ra che mất cổ tay hoặc cẳng tay
    đang đút vào túi quần của người mẫu.
    """
    if candidate_mask is None or pose_context is None:
        return 1.0

    keypoints = pose_context.get("keypoints")
    confidences = pose_context.get("confidences")
    if keypoints is None or confidences is None or len(keypoints) < 11:
        return 1.0

    mask = (candidate_mask > 127).astype(np.uint8)
    h, w = mask.shape[:2]

    shoulder_span = abs(float(keypoints[6][0] - keypoints[5][0]))
    wrist_radius = max(14, int(round(max(60.0, shoulder_span) * 0.08)))
    forearm_thickness = max(12, int(round(max(60.0, shoulder_span) * 0.09)))

    label = (garment_label or "").lower()
    allows_forearm_coverage = any(
        word in label for word in ["long_sleeved", "coat", "jacket", "sweater"]
    )
    # Với trang phục dài tay, mask phủ khuỷu/cổ tay là đúng thiết kế chứ không phải artefact.
    if allows_forearm_coverage:
        return 1.0
    is_short_sleeve = True

    total_penalty = 0.0

    # Kiểm tra cả 2 tay: (elbow, wrist) = (7, 9) tay trái, (8, 10) tay phải
    for elbow_idx, wrist_idx in ((7, 9), (8, 10)):
        # 1. Kiểm tra vùng cổ tay (người mẫu đút tay vào túi quần)
        if confidences[wrist_idx] >= 0.25:
            wx, wy = int(round(keypoints[wrist_idx][0])), int(round(keypoints[wrist_idx][1]))
            wrist_roi = np.zeros((h, w), dtype=np.uint8)
            cv2.circle(wrist_roi, (wx, wy), wrist_radius, 255, -1)

            overlap_pixels = np.count_nonzero((wrist_roi > 0) & (mask > 0))
            roi_pixels = np.count_nonzero(wrist_roi > 0)
            if roi_pixels > 0:
                overlap_ratio = overlap_pixels / roi_pixels
                if overlap_ratio > 0.05:
                    # Phạt tỷ lệ thuận với diện tích áo trùm lên cổ tay
                    total_penalty += min(1.0, overlap_ratio * 2.0) * 0.50

        # 2. Với áo ngắn tay/polo: kiểm tra nửa dưới cẳng tay (từ 35% khuỷu tay đến cổ tay)
        if is_short_sleeve and confidences[elbow_idx] >= 0.25 and confidences[wrist_idx] >= 0.25:
            ex, ey = keypoints[elbow_idx]
            wx, wy = keypoints[wrist_idx]
            p_start = (int(round(ex + 0.35 * (wx - ex))), int(round(ey + 0.35 * (wy - ey))))
            p_end = (int(round(wx)), int(round(wy)))

            forearm_roi = np.zeros((h, w), dtype=np.uint8)
            cv2.line(forearm_roi, p_start, p_end, 255, forearm_thickness, cv2.LINE_AA)

            forearm_overlap = np.count_nonzero((forearm_roi > 0) & (mask > 0))
            forearm_total = np.count_nonzero(forearm_roi > 0)
            if forearm_total > 0:
                forearm_ratio = forearm_overlap / forearm_total
                if forearm_ratio > 0.05:
                    # Phạt nếu vạt áo/tay áo phình sang hai bên đè lên cẳng tay
                    total_penalty += min(1.0, forearm_ratio * 2.0) * 0.50

    return float(np.clip(1.0 - total_penalty, 0.0, 1.0))
def evaluate_dress_proportions_and_color(candidate_image, candidate_mask, pose_context, reference_image, reference_mask):
    """
    Tiêu chí chuyên biệt cho Dress:
    1. Phạt nếu váy phình to bất thường so với khoảng cách hai hông (hip_span).
    2. Phạt nếu váy bị loang mảng màu lạ không có trong đầm mẫu (chống hallucination).
    """
    if candidate_mask is None or pose_context is None:
        return 0.5, 0.5

    keypoints = pose_context.get("keypoints")
    confidences = pose_context.get("confidences")
    if keypoints is None or len(keypoints) < 13:
        return 0.5, 0.5

    mask = (candidate_mask > 127).astype(np.uint8)
    ys, xs = np.where(mask > 0)
    if len(xs) < 100:
        return 0.0, 0.0

    # 1. Đo tỷ lệ phình ngang của váy so với khoảng cách hai hông
    # Keypoint 11, 12 là Left Hip và Right Hip
    hip_width = abs(float(keypoints[12][0] - keypoints[11][0]))
    if hip_width < 20.0:
        # Nếu hông bị che, dùng 70% bề rộng vai
        hip_width = max(30.0, abs(float(keypoints[6][0] - keypoints[5][0])) * 0.75)

    dress_width = float(xs.max() - xs.min())
    width_ratio = dress_width / hip_width

    # Tỷ lệ lý tưởng cho đầm suông/chữ A thường trong khoảng 1.1 - 1.45 lần hông
    fit_penalty = 0.0
    if width_ratio > 1.45:
        # Váy phình quá to sang 2 bên
        fit_penalty = min(1.0, (width_ratio - 1.45) / 0.6)
    dress_fit_score = float(np.clip(1.0 - fit_penalty, 0.0, 1.0))

    # 2. Kiểm tra độ tương đồng màu sắc & chống loang màu lạ (màu đỏ mận)
    color_score = garment_color_similarity(
        reference_image, reference_mask, candidate_image, candidate_mask
    )

    return dress_fit_score, color_score

def garment_color_similarity(reference_image, reference_mask, candidate_image, candidate_mask):
    """So màu trong vùng garment bằng histogram Lab; 1.0 là gần màu gốc nhất."""
    if reference_mask is None or candidate_mask is None:
        return 0.0

    def masked_lab_histogram(image, mask):
        image_np = np.asarray(image.convert("RGB"), dtype=np.uint8)
        mask = (mask > 127).astype(np.uint8) * 255
        # Co mask nhẹ để da/nền ở rìa garment không làm lệch màu.
        clean_mask = cv2.erode(mask, np.ones((5, 5), np.uint8), iterations=1)
        if np.count_nonzero(clean_mask) < 50:
            clean_mask = mask
        if np.count_nonzero(clean_mask) < 50:
            return None
        lab_image = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
        histogram = cv2.calcHist(
            [lab_image], [0, 1, 2], clean_mask,
            [8, 12, 12], [0, 256, 0, 256, 0, 256],
        )
        return cv2.normalize(
            histogram, None, alpha=1.0, norm_type=cv2.NORM_L1
        )

    reference_histogram = masked_lab_histogram(reference_image, reference_mask)
    candidate_histogram = masked_lab_histogram(candidate_image, candidate_mask)
    if reference_histogram is None or candidate_histogram is None:
        return 0.0

    distance = cv2.compareHist(
        reference_histogram, candidate_histogram, cv2.HISTCMP_BHATTACHARYYA
    )
    return float(np.clip(1.0 - distance, 0.0, 1.0))


DRESS_STYLE_FAMILY = {
    "vest_dress": "sleeveless_dress",
    "sling_dress": "sleeveless_dress",
    "short_sleeved_dress": "short_sleeved_dress",
    "long_sleeved_dress": "long_sleeved_dress",
}


def candidate_match_quality(candidate_label, target_label, expected_category):
    """Trả mức khớp subtype; confidence chỉ được dùng khi hai candidate cùng mức."""
    candidate_label = (candidate_label or "").lower()
    target_label = (target_label or "").lower()
    if not candidate_label:
        return 0, "không nhận diện đúng category"
    if candidate_label == target_label:
        return 3, "khớp subtype chính xác"

    if expected_category == "dress":
        candidate_family = DRESS_STYLE_FAMILY.get(candidate_label)
        target_family = DRESS_STYLE_FAMILY.get(target_label)
        if candidate_family and candidate_family == target_family:
            return 2, f"cùng nhóm kiểu tay ({target_family})"
        return 1, "đúng dress nhưng sai kiểu tay/subtype"

    if expected_category == "lower_body":
        # DeepFashion2 có thể nhận lại quần/shorts đang mặc hoặc nhầm giữa skirt,
        # shorts và trousers trên ảnh người. Đây vẫn là đúng vùng lower-body nên
        # chỉ hạ ưu tiên và cảnh báo, không loại candidate khỏi kết quả try-on.
        return (
            1,
            f"đúng lower-body nhưng detector nhận '{candidate_label}' thay vì '{target_label}'",
        )

    return 1, "đúng upper-body nhưng khác subtype"


# Cấu hình ổn định dùng chung cho hai checkpoint OOTDiffusion VITON-HD/DressCode.
OOT_N_SAMPLES = 4
OOT_N_STEPS = 20
OOT_IMAGE_SCALE = 2.0
OOT_SEED = 42  # Cố định để so sánh thí nghiệm; đổi thành -1 nếu muốn biến thể ngẫu nhiên.
OOT_MANUAL_CANDIDATE = None  # Có thể đặt 1..4 sau khi xem bảng candidate rồi chạy lại cell.

print(
    f"[*] Cấu hình OOTDiffusion: samples={OOT_N_SAMPLES}, steps={OOT_N_STEPS}, "
    f"scale={OOT_IMAGE_SCALE}, seed={OOT_SEED}"
)

def validate_person_frame_for_category(selected_frame, selected_category, garment_subtype):
    """Áp dụng progressive validation: ưu tiên cảnh báo thay vì chặn khi vẫn có thể thử đồ."""
    if selected_frame == "invalid":
        raise ValueError("Ảnh không thấy rõ vai và hông nên chưa đủ điều kiện try-on.")

    if selected_category == "upper_body":
        return "accepted", "Khung ảnh đủ cho thử áo."

    if selected_category == "dress":
        if selected_frame == "full_body":
            return "accepted", "Ảnh toàn thân phù hợp để thử mọi chiều dài đầm."
        if selected_frame == "three_quarter":
            return (
                "accepted_with_warning",
                "Ảnh 3/4 vẫn được thử đầm; phù hợp nhất với đầm ngắn hoặc đầm trên gối. "
                "Phần đầm dài dưới vùng ảnh có thể không được thể hiện đầy đủ.",
            )
        return (
            "accepted_with_warning",
            "Ảnh chỉ thấy đến hông/đùi nhưng vẫn được thử đầm ở chế độ giới hạn. "
            "Kết quả chỉ đáng tin trong vùng cơ thể đang nhìn thấy; nên dùng ảnh thấy ít nhất đến đầu gối "
            "nếu muốn đánh giá phom và chiều dài đầm.",
        )

    # Lower-body: shorts/skirt có thể chạy giới hạn với ảnh 3/4 hoặc half-body; trousers vẫn cần thấy mắt cá chân.
    subtype = (garment_subtype or "").lower()
    if selected_frame == "full_body":
        return "accepted", "Ảnh toàn thân phù hợp để thử lower-body."
    if selected_frame in {"three_quarter", "half_body"} and subtype in {"shorts", "skirt"}:
        if selected_frame == "three_quarter":
            message = (
                f"Ảnh 3/4 đủ để thử {subtype}, nhưng chỉ nên đánh giá phần trang phục nằm trong vùng ảnh."
            )
        else:
            message = (
                f"Ảnh half-body vẫn được thử {subtype} ở chế độ giới hạn vì đã thấy hông/đùi. "
                "Kết quả chỉ đáng tin trong vùng cơ thể đang hiển thị; nên dùng ảnh thấy đầu gối để đánh giá "
                "đầy đủ chiều dài và phom trang phục."
            )
        return "accepted_with_warning", message
    raise ValueError(
        f"Ảnh '{selected_frame}' chưa đủ vùng chân để thử '{subtype or selected_category}'. "
        "Quần dài cần thấy mắt cá; chân váy/quần short cần thấy ít nhất một đầu gối."
    )


frame_validation_status, frame_validation_message = validate_person_frame_for_category(
    person_frame, garment_category, garment_class_name
)
if frame_validation_status == "accepted_with_warning":
    print(f"[!] Khung người được chấp nhận có điều kiện: {frame_validation_message}")
else:
    print(f"[+] Kiểm tra khung người/category hợp lệ: {frame_validation_message}")

if garment_category == "dress":
    print(
        "[i] Hướng dẫn thử đầm: có thể dùng ảnh đang mặc áo + quần, nhưng nên chọn "
        "trang phục ôm gọn (áo hai dây/áo ôm, legging, jean ôm, short ôm hoặc đồ tập). "
        "Tránh áo khoác, áo dài tay rộng, quần ống rộng và tay che trước eo/thân."
    )
elif garment_category == "lower_body":
    lower_guidance = {
        "shorts": (
            "nên mặc short ôm hoặc legging ngắn; tránh quần dài, váy dài và tay che hông"
        ),
        "skirt": (
            "nên mặc short/legging ôm hoặc chân váy gọn có chiều dài tương tự; "
            "tránh quần ống rộng, váy xòe nhiều lớp và tay che eo"
        ),
        "trousers": (
            "nên mặc legging, jean ôm hoặc quần dáng gọn; tránh váy dài, quần ống rộng "
            "và chân bắt chéo"
        ),
    }.get(garment_class_name, "nên mặc phần dưới ôm gọn và không che hông/chân")
    print(f"[i] Hướng dẫn thử lower-body: {lower_guidance}.")

# Routing OOTDiffusion: VITON-HD xử lý upper-body; DressCode xử lý lower-body và dress.
# Frame chỉ quyết định mức chấp nhận/cảnh báo, không tự đổi endpoint.
try:
    from gradio_client import Client, handle_file

    def create_space_client(space_id):
        if not hf_token:
            return Client(space_id)
        try:
            return Client(space_id, token=hf_token)
        except TypeError:
            # Tương thích gradio_client phiên bản cũ (trước khi hf_token đổi tên thành token).
            return Client(space_id, hf_token=hf_token)

    client = create_space_client("levihsu/OOTDiffusion")
    prepared_person_path = str(
        Path(tryon_dir) / "person_oot_768x1024.png"
    )

    prepared_person_path, person_resize_metadata = prepare_oot_person(
        source_path=person_path,
        output_path=prepared_person_path,
    )
    prepared_person_image = Image.open(prepared_person_path).convert("RGB")

    # Ảnh/mask tham chiếu dùng để đo độ trung thành màu của từng candidate.
    if os.path.exists(out_nobg_path):
        garment_reference_rgba = Image.open(out_nobg_path).convert("RGBA")
        garment_reference_image = garment_reference_rgba.convert("RGB")
        garment_reference_mask = np.asarray(
            garment_reference_rgba.getchannel("A"), dtype=np.uint8
        )
    else:
        garment_reference_image = Image.open(garment_for_tryon_path).convert("RGB")
        _, _, garment_reference_mask = detect_output_garment(
            garment_reference_image, garment_category
        )

    pose_context = detect_person_pose_context(prepared_person_image)
    protected_body_mask = None
    if garment_category in {"upper_body", "dress"}:
        protected_body_mask = upper_body_protected_mask(
            prepared_person_image.size, pose_context, garment_class_name
        )

    source_garment_mask = None
    if garment_category == "dress":
        # Người dùng có thể đang mặc áo + quần/váy rời; hợp các mask đồ cũ để đầm mới
        # được phép thay toàn bộ trang phục nhưng không được vẽ lại cơ thể và nền.
        source_masks = []
        for source_category in ("dress", "upper_body", "lower_body"):
            _, _, detected_mask = detect_output_garment(
                prepared_person_image, source_category
            )
            if detected_mask is not None:
                source_masks.append(detected_mask)
        if source_masks:
            source_garment_mask = np.maximum.reduce(source_masks).astype(np.uint8)
    else:
        _, _, source_garment_mask = detect_output_garment(
            prepared_person_image, garment_category
        )

    if garment_category in {"upper_body", "lower_body", "dress"}:
        if source_garment_mask is None:
            print("[!] Không tách được mask đồ đang mặc; điểm bảo toàn ngoài garment sẽ giảm độ tin cậy.")
        else:
            print("[+] Đã tạo mask đồ đang mặc để bảo toàn cơ thể và nền ngoài garment.")

    def run_ootdiffusion(selected_category):
        common_inputs = {
            "vton_img": handle_file(prepared_person_path),
            "garm_img": handle_file(garment_for_tryon_path),
            "n_samples": OOT_N_SAMPLES,
            "n_steps": OOT_N_STEPS,
            "image_scale": OOT_IMAGE_SCALE,
            "seed": OOT_SEED,
        }

        if selected_category == "upper_body":
            backend = "OOTDiffusion VITON-HD (Upper-body)"
            print(f"[*] {selected_category} -> {backend} [/process_hd]")
            # /process_hd cố định upper-body và không nhận tham số category.
            response = client.predict(
                **common_inputs,
                api_name="/process_hd",
            )
            return response, backend

        oot_category = {
            "lower_body": "Lower-body",
            "dress": "Dress",
        }[selected_category]
        backend = f"OOTDiffusion DressCode ({oot_category})"
        print(f"[*] {selected_category} -> {backend} [/process_dc]")
        response = client.predict(
            **common_inputs,
            category=oot_category,
            api_name="/process_dc",
        )
        return response, backend

    result, tryon_backend = run_ootdiffusion(garment_category)

    result_img_paths = extract_gradio_image_paths(result)
    if not result_img_paths:
        raise RuntimeError(f"Không đọc được đường dẫn ảnh từ kết quả Gradio: {result!r}")

    candidate_reports = []
    candidate_images = []
    eligible_candidates = []
    for candidate_index, candidate_path in enumerate(result_img_paths, start=1):
        candidate_image = Image.open(candidate_path).convert("RGB")
        candidate_label, candidate_confidence, candidate_garment_mask = detect_output_garment(
            candidate_image, garment_category
        )
        match_level, match_reason = candidate_match_quality(
            candidate_label, garment_class_name, garment_category
        )
        color_similarity = garment_color_similarity(
            garment_reference_image, garment_reference_mask,
            candidate_image, candidate_garment_mask,
        )
        edit_mask = None
        outside_change = 0.0
        outside_preservation = 1.0
        body_change = 0.0
        body_preservation = 1.0
        
        protected_candidate_image = candidate_image
        if garment_category in {"upper_body", "lower_body", "dress"}:
            # Chỉ lấy vùng garment mới từ candidate; mọi pixel ngoài vùng đồ lấy từ ảnh người gốc.
            edit_mask = garment_edit_mask(
                source_garment_mask, candidate_garment_mask
            )
            outside_change = outside_garment_change_score(
                candidate_image, prepared_person_image, edit_mask
            )
            outside_preservation = change_to_preservation_score(outside_change)
            # Pixel-wise body comparison chỉ phù hợp với dress. VITON-HD upper-body có thể
            # dịch chuyển tay vài pixel dù kết quả hợp lệ; dùng metric này làm hard gate sẽ loại nhầm.
            if garment_category == "dress":
                body_change, body_preservation = protected_region_artifact_score(
                    candidate_image, prepared_person_image, protected_body_mask
                )
            # VITON-HD đã sinh upper-body hoàn chỉnh. Composite với union mask áo cũ/mới
            # sẽ kéo tay áo và vạt áo cũ trở lại, tạo các mảng trắng chồng lên áo mới.
            # Vì vậy upper-body giữ nguyên candidate thô; lower-body/dress mới cần composite.
            if garment_category != "upper_body":
                protected_candidate_image = composite_preserving_source(
                    candidate_image, prepared_person_image, edit_mask,
                    protected_mask=protected_body_mask if garment_category == "dress" else None,
                )

        fit_score = garment_fit_score(candidate_garment_mask, pose_context, garment_category, garment_class_name)
        # Tính điểm bảo toàn chi thể (chống che mất tay đút túi quần)
        limb_integrity = calculate_limb_integrity_score(
            candidate_garment_mask, pose_context, garment_class_name
        )

        # if garment_category == "upper_body":
        #     # Ưu tiên cao nhất (35%) cho độ toàn vẹn chi thể không bị áo che khuất
        #     quality_score = (
        #         0.25 * color_similarity
        #         + 0.35 * limb_integrity        # <--- TIÊU CHÍ MỚI
        #         + 0.25 * fit_score
        #         + 0.15 * candidate_confidence
        #     )
        # else:
        #     quality_score = (
        #         0.20 * color_similarity
        #         + 0.25 * outside_preservation
        #         + 0.25 * body_preservation
        #         + 0.25 * fit_score
        #         + 0.05 * candidate_confidence
        #     )
        dress_width_fit = 1.0
        if garment_category == "dress":
            dress_width_fit, color_similarity = evaluate_dress_proportions_and_color(
                candidate_image, candidate_garment_mask, pose_context,
                garment_reference_image, garment_reference_mask
            )
            # Dress phải giữ nguyên tay/cơ thể và nền; màu đẹp không được bù cho artefact ngoài garment.
            quality_score = (
                0.30 * color_similarity
                + 0.15 * dress_width_fit
                + 0.15 * outside_preservation
                + 0.15 * body_preservation
                + 0.10 * limb_integrity
                + 0.10 * fit_score
                + 0.05 * candidate_confidence
            )
        elif garment_category == "upper_body":
            quality_score = (
                0.25 * color_similarity
                + 0.35 * limb_integrity
                + 0.25 * fit_score
                + 0.15 * candidate_confidence
            )
        else: # lower_body
            quality_score = (
                0.20 * color_similarity
                + 0.25 * outside_preservation
                + 0.25 * body_preservation
                + 0.25 * fit_score
                + 0.05 * candidate_confidence
            )

        artifact_reasons = []
        outside_minimum = {"upper_body": 0.72, "lower_body": 0.65, "dress": 0.65}[garment_category]
        if outside_preservation < outside_minimum:
            artifact_reasons.append(
                f"ngoài garment thấp ({outside_preservation:.3f} < {outside_minimum:.2f})"
            )
        if garment_category == "dress" and body_preservation < 0.30:
            artifact_reasons.append(f"tay/cơ thể đổi màu ({body_preservation:.3f})")
        limb_minimum = {"upper_body": 0.65, "dress": 0.75}.get(garment_category)
        if limb_minimum is not None and limb_integrity < limb_minimum:
            artifact_reasons.append(
                f"mask garment lấn vào tay ({limb_integrity:.3f} < {limb_minimum:.2f})"
            )
        artifact_ok = not artifact_reasons
        artifact_status = "đạt artifact gate" if artifact_ok else "LOẠI: " + "; ".join(artifact_reasons)

        candidate_reports.append(
            f"#{candidate_index}: detector_label={candidate_label or 'không nhận diện'} "
            f"({candidate_confidence:.2f}; {match_reason}; "
            f"màu={color_similarity:.3f}; ngoài={outside_preservation:.3f}; "
            f"cơ thể={body_preservation:.3f}; fit={fit_score:.3f}; "
            f"tổng={quality_score:.3f}; {artifact_status})"
        )
        print(f"    Candidate {candidate_reports[-1]}")

        candidate_info = {
            "index": candidate_index,
            "image": protected_candidate_image,
            "raw_image": candidate_image,
            "label": candidate_label,
            "confidence": candidate_confidence,
            "color_similarity": color_similarity,
            "limb_integrity": limb_integrity,    # <--- THÊM DÒNG NÀY
            "outside_change": outside_change,
            "outside_preservation": outside_preservation,
            "body_change": body_change,
            "body_preservation": body_preservation,
            "fit_score": fit_score,
            "quality_score": quality_score,
            "match_level": match_level,
            "match_reason": match_reason,
            "artifact_ok": artifact_ok,
            "artifact_status": artifact_status,
            "eligible": match_level > 0 and artifact_ok,
        }
        candidate_images.append(candidate_info)
        if candidate_info["eligible"]:
            eligible_candidates.append(candidate_info)

    if not eligible_candidates:
        raise RuntimeError(
            f"Không candidate nào vượt qua quality/artifact gate cho vùng "
            f"'{garment_category}' sau {len(result_img_paths)} kết quả (detector có thể vẫn nhận diện đúng): "
            f"{', '.join(candidate_reports)}"
        )

    if OOT_MANUAL_CANDIDATE is not None:
        manual_matches = [
            item for item in eligible_candidates
            if item["index"] == OOT_MANUAL_CANDIDATE
        ]
        if not manual_matches:
            raise ValueError(
                f"OOT_MANUAL_CANDIDATE={OOT_MANUAL_CANDIDATE} không hợp lệ hoặc sai category."
            )
        selected_candidate = manual_matches[0]
    else:
        # # Tầng 1: chỉ so các candidate có mức khớp subtype tốt nhất hiện có.
        # best_match_level = max(item["match_level"] for item in eligible_candidates)
        # selection_pool = [
        #     item for item in eligible_candidates
        #     if item["match_level"] == best_match_level
        # ]
        # # Tầng 2: điểm tổng hợp cân bằng màu, bảo toàn cơ thể/nền và độ fitting.
        # selected_candidate = max(
        #     selection_pool,
        #     key=lambda item: (
        #         item["quality_score"],
        #         item["confidence"],
        #     ),
        # )
                # Với Dress: Nếu candidate duy nhất có nhãn subtype lại bị màu quá thấp (< 0.25),
        # lập tức mở rộng pool để chọn candidate có màu sắc và độ vừa vặn thực tế tốt nhất.
        if garment_category == "dress":
            # Ưu tiên ứng viên có màu sắc trung thực với đầm shop và form dáng gọn
            selected_candidate = max(
                eligible_candidates,
                key=lambda item: item["quality_score"]
            )
        else:
            best_match_level = max(item["match_level"] for item in eligible_candidates)
            selection_pool = [
                item for item in eligible_candidates
                if item["match_level"] == best_match_level
            ]
            selected_candidate = max(
                selection_pool,
                key=lambda item: (item["quality_score"], item["confidence"])
            )


    output = selected_candidate["image"]
    selected_index = selected_candidate["index"]
    print(
        f"[+] Chọn candidate #{selected_index}: detector_label={selected_candidate['label']} "
        f"(tổng={selected_candidate['quality_score']:.3f}, "
        f"màu={selected_candidate['color_similarity']:.3f}, "
        f"ngoài={selected_candidate['outside_preservation']:.3f}, "
        f"cơ thể={selected_candidate['body_preservation']:.3f}, "
        f"fit={selected_candidate['fit_score']:.3f}, "
        f"confidence={selected_candidate['confidence']:.2f})."
    )
    print(f"    Lý do chọn: {selected_candidate['match_reason']}.")
    if selected_candidate["match_level"] < 3:
        print(
            f"[!] Không có candidate khớp chính xác '{garment_class_name}'; "
            "đang dùng phương án gần nhất. Hãy kiểm tra trực quan trước khi chấp nhận."
        )

    # Luôn hiển thị toàn bộ candidate để có thể đánh giá artefact bằng mắt.
    fig, axes = plt.subplots(1, len(candidate_images), figsize=(5 * len(candidate_images), 6))
    axes = np.atleast_1d(axes).ravel()
    for axis, item in zip(axes, candidate_images):
        chosen = item["index"] == selected_index
        exact_match = item["match_level"] == 3
        chosen_text = " - ĐÃ CHỌN" if chosen and exact_match else (" - TẠM CHỌN" if chosen else "")
        # Hiển thị candidate thô để đánh giá đúng ảnh do backend sinh trước hậu xử lý.
        axis.imshow(item["image"])
        axis.set_title(
            f"Candidate #{item['index']}{chosen_text}" +
            f"\nDetector: {item['label'] or 'không nhận diện'} ({item['confidence']:.2f})" +
            f"\nTổng: {item['quality_score']:.3f} | Màu: {item['color_similarity']:.3f}" +
            f"\nNgoài: {item['outside_preservation']:.3f} | Cơ thể: {item['body_preservation']:.3f} | Fit: {item['fit_score']:.3f}" +
            f"\nTay: {item['limb_integrity']:.3f} | Conf: {item['confidence']:.2f}" +
            f"\n{item['artifact_status']}",
            color=("green" if exact_match else "darkorange") if chosen else "black",
            fontweight="bold" if chosen else "normal",
        )

        axis.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"[+] {tryon_backend} sinh ảnh thành công!")
except Exception as e:
    remote_error = e
    print(f"[-] Backend try-on cho '{garment_category}' thất bại: {e}")

if output is None:
    raise RuntimeError(
        f"Không thể chạy OOTDiffusion cho {garment_category}: {remote_error}. "
        "Notebook đã dừng để không lưu ảnh preview giả làm kết quả VTON. "
        "Nếu request chưa sinh được ảnh, hãy kiểm tra HF_TOKEN/quota; nếu các candidate đều bị loại, "
        "hãy xem lý do quality/artifact gate ở log rồi chạy lại Bước 6."
    )

infer_time = time.time() - start_time

# Khôi phục đúng tỷ lệ/kích thước ảnh người trước khi lưu và lập báo cáo.
output = restore_oot_result(
    output,
    person_resize_metadata,
)
# LƯU RIÊNG ẢNH KẾT QUẢ THỬ ĐỒ (ẢNH 1/2 TRONG results/tryon/)
output.save(tryon_out_path, "PNG")
print(f"[+] Hoàn thành thử đồ AI sau: {infer_time:.2f} giây!")
print(f"✅ [1/2] Đã lưu RIÊNG ảnh thử đồ ảo (Single Try-on) vào:")
print(f"    -> {os.path.abspath(tryon_out_path)} ({os.path.getsize(tryon_out_path)/1024:.1f} KB)")

# Chuẩn bị các biến cho Bước 7
human_img = Image.open(person_path).convert("RGB")
cloth_img = Image.open(garment_for_tryon_path).convert("RGB")


### Bước 7: Xuất bảng ảnh đối sánh nghiệm thu Tuần 4


In [ ]:
import matplotlib.pyplot as plt

report_compare_path = "data/test_samples/results/tryon/week4_report_comparison.jpg"
os.makedirs(os.path.dirname(report_compare_path), exist_ok=True)

fig, ax = plt.subplots(1, 3, figsize=(18, 8))
ax[0].imshow(human_img); ax[0].set_title("1. Người mẫu gốc (YoFaSo User)", fontsize=14, fontweight="bold")
ax[1].imshow(cloth_img); ax[1].set_title("2. Trang phục shop (YoFaSo Garment)", fontsize=14, fontweight="bold")
ax[2].imshow(output); ax[2].set_title(f"3. Kết quả thử đồ AI ({tryon_backend} - {infer_time:.2f}s)", fontsize=14, fontweight="bold")
for a in ax: a.axis('off')
plt.tight_layout()
plt.savefig(report_compare_path, dpi=200, bbox_inches='tight')
plt.show()

print("="*75)
print("             TỔNG KẾT KẾT QUẢ THỬ NGHIỆM TUẦN 4 (YoFaSoShop_VTON)             ")
print("="*75)
print(f"- Trạng thái bóc tách nền: THÀNH CÔNG (DeepFashion2 YOLOv8-seg)")
print(f"- Backend try-on: {tryon_backend}")
print(f"- Category: {garment_category}; khung người: {person_frame}; xác thực: {frame_validation_status}; thời gian render: {infer_time:.2f} giây")
print(f"- Thư mục results/segmented/: Đã lưu garment mask, RGBA và garment nền trắng")
print(f"- Thư mục results/tryon/    : Đã lưu đủ 2 ảnh kết quả:")
print(f"    [1] Ảnh thử đồ đơn lẻ : {os.path.abspath(tryon_out_path)}")
print(f"    [2] Bảng đối sánh 3 cột: {os.path.abspath(report_compare_path)}")
print("="*75)


### Bước 8: Tải trọn bộ kết quả về máy tính cá nhân (Dành riêng cho Google Colab)
Nếu bạn chạy trên **Google Colab**, toàn bộ ảnh được lưu trữ trên máy chủ đám mây của Google. Chạy cell bên dưới để tự động nén toàn bộ thư mục `results/` và tải file `.zip` về thư mục **Downloads** trên máy tính của bạn!

In [ ]:
import shutil, os

results_dir = "data/test_samples/results"
zip_base_name = "yofaso_vton_week4_results"
zip_file = f"{zip_base_name}.zip"

# Nén toàn bộ thư mục kết quả
if os.path.exists(results_dir):
    shutil.make_archive(zip_base_name, 'zip', results_dir)
    print(f"📦 Đã nén thành công: {zip_file} ({os.path.getsize(zip_file)/1024:.1f} KB)")
    
    # Kích hoạt tải về tự động nếu chạy trên Colab
    try:
        from google.colab import files
        print("🚀 Đang gửi yêu cầu tải file zip về trình duyệt của bạn...")
        files.download(zip_file)
        print("👉 Bạn chỉ cần kiểm tra thư mục Downloads trên máy tính và giải nén dán vào data/test_samples/results/")
    except ImportError:
        print(f"💻 Bạn đang chạy cục bộ (Local). Toàn bộ ảnh đã được lưu trực tiếp tại:")
        print(f"   {os.path.abspath(results_dir)}")
else:
    print(f"⚠️ Chưa tìm thấy thư mục {results_dir} để đóng gói!")
